# Inception V1 2015

## Objective
1. To a gain a better understanding of Inception image classification models introdued in the paper **[Going deeper with convolutions](https://arxiv.org/pdf/1409.4842)**

**Note**:
- To look at the entire model's implementation at one place you can have a look at the model's code written here`computer-vision/src/computer_vision/classification/vgg.py`
- To understand image classification thoroughly look at the information provided under the LeNet 1 notebook `computer-vision/src/computer_vision/classification/lenet_1.ipynb`
- To understand dropout layers in depth you can look at [Dropout Layer Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/dropout_layers.ipynb)
- To understand pooling layers in depth you can look at [Pooling Layers Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/pooling_layers.ipynb)
- To understand what batch normalization is in depth you can look at [Batch Normalization Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/batch_normalization.ipynb)
- To understand convolution neural network layer in depth you can look at [Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb) (**I advised you to go through this notebook before you go ahead for better understanding of convolution layers**)

## Introduction
The most straight forward way of improving a deep learning model's capability has been by increasing the model's size. The increase in size can be done in two ways. First is increasing the number of layers of the model and the second is increasing the number of units per level. Increasing them uniformly comes with 2 drawbacks.

1. Increasing the size of the model leads to overfitting as the model's total number of training parameters would outgrow the training samples. Building a large dataset of good quality samples is a difficult task.
2. Increasing ths size of the model increasing the computational load. For example if we are building a deep learning model where we stack two convolution layers together and increase the number of feature maps by two it will lead to a quadratic increase in computation. If added computation load is used inefficiently (most of the weights turn out to be zero) it will lead to wastage of computation that would have helped in learning better representations of the underlying data.

Till now models have simply increased the depth and width of the model to produce large models in order to get better at predicting the right class from an image. But this means these large models cannot be deployed to produce fruitful work. The aim of the authors was to build a model that would allow them to deploy these models on edge devices to perform computer vision related tasks.

The authors decided to research a structure that would have computational complexity less than equal to that of **1.5 billion multi adds at inference time** which allows them to deploy models on edge devices.

## Concepts Introduced in Inception V1 model
The inception model popularly known as **GoogleNet** introduced 2 concepts namely,
1. Inception Module
2. auxilary Classifiers

### Inception Module
In previous models discussed and produced such as AlexNet, VGG models when trained on dataset have seen weights across multiple layers to be zero or close to zero. These weights are **dead** weights as they do not learn and extract any useful information from the data but they participate in every forward pass which is highly inefficient. An alternative to this is to use a sparse model where we drop these **dead** weights and only utilize the weights which are non zero to participate. The current CPU and GPU architecture does not allow for that to be done efficiently.

There has been research done where it has been found that to get the same performance from a sparse model and also utilize the current CPU and GPU capabilities we can  construct a structure layer by layer by analyzing the correlation statistics of the activations of the last layer and clustering neurons with highly correlated outputs. This requires strong mathematical proofs and calculations in order to get these neurons clustered.

To build such a structure the authors decided to research if such a structure is possible to be constructed using Convolution Neural Networks. The result of this research has been the **inception** module. This module structure came about in two iterations

#### 1. Iteration 1: Naive Module
As discussed previously that previous models have been made such that they increase the size of the model layer by layer increase the depth and width uniformly. This is done by reducing the size of the feature maps to  reduce the computational load that comes by performing convolution operations across large feature map size and increasing the number of feature maps per layer to compensate for the loss of information caused due to drop in the size of the feature maps.

In this process what we are doing is we are **extracting correlation information across a fixed receptive field per layer or per n number of layers** before we reduce the size of the feature maps. The authors decided to make the computational load be more efficient we should extract correlation information spread across different receptive fields within the same layer. This would allow information spread across different fields to participate in next layers information extraction allowing the model to extract more information by correlating information present across a small receptive field with the information spread over a larger receptive field. To do so the authors made use of 1x1 convolution layers to extract information from local smaller receptive fields and 3x3 and 5x5 convolution layers to extract information spread across larger receptive fields. The authors also used 3x3 **Max Pooling** layers as layers that would also allow them to extract information. Max Pooling layers were added because these layers have helped previous models to improve their performance and the author's did not want to miss out on the performance gains that these layers provide. 

The way these layers are structured is that the input to this layer is passed through 1x1, 3x3, 5x5 and Max Pooling layers independently and the results are concatenated across the channels. This concatenated output becomes the input to the next layer allowing them to have information present across smaller and larger receptive fields at once. This is computationally efficent to some extent when it comes to extracting information spread across different receptive fields and providing it to the next layer in one single layer. This module is name as **Naive Inception Module**. This is how it looks
```text
                           [Filter Concatenation]
                                     ▲
           ┌─────────────────┬───────┴───────┬─────────────────┐
           │                 │               │                 │
           │                 │               │                 │
[1x1 convolutions] [3x3 convolutions] [5x5 convolutions] [3x3 max pooling]
           ▲                 ▲               ▲                 ▲
           │                 │               │                 │
           └─────────────────┴───────┬───────┴─────────────────┘
                                     │
                             [Previous layer]
```

Here we are parsing all feature maps through each type of convolution layers and pool layers.Though computationally efficient as it provides information spread across different receptive fields. It is still computationally expensive when compared to its improved version. In the naive module all the feature maps part of the input propapagate through 1x1, 3x3 and 5x5 convolution filters. Performing convolution operations with filters with large receptive fields is computationally expensive. We can see this using an example where we compare two versions of the naive inception module where we have an input image of shape $(1,3,224,224)$ which has been down sampled to $(1,192,56,56)$ which we pass through two different configurations of naive inception modules. The convolution filters
1. 32 5x5 convolution filters
   1. 64 1x1 convolution filters
   2. 128 3x3 convolution filters
   3. 32 5x5 convolution filters
   4. All input feature maps will go through this layer
2. 64 5x5 convolution filters
   1. 64 1x1 convolution filters
   2. 128 3x3 convolution filters
   3. 64 5x5 convolution filters
   4. All input feature maps will go through this layer

The configurations differ by the change in number of convolution filters to be used by the **5x5** convolution filter.

In [1]:
from torchinfo import summary
from computer_vision.classification.inceptionnet_v1 import InceptionNaiveModule, InceptionModule, InceptionNetV1

##### Case 1: 32 5x5 convolution filters

In [2]:
summary(InceptionNaiveModule(in_channels=192,
                             out_channels_1x1=32,
                             out_channels_3x3=128,
                             out_channels_5x5=32), 
                             input_size=(1, 192, 56, 56), 
                             col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
InceptionNaiveModule                     [1, 192, 56, 56]          [1, 384, 56, 56]          --                        --
├─Sequential: 1-1                        [1, 192, 56, 56]          [1, 32, 56, 56]           --                        --
│    └─Conv2d: 2-1                       [1, 192, 56, 56]          [1, 32, 56, 56]           [1, 1]                    6,144
│    └─ReLU: 2-2                         [1, 32, 56, 56]           [1, 32, 56, 56]           --                        --
├─Sequential: 1-2                        [1, 192, 56, 56]          [1, 128, 56, 56]          --                        --
│    └─Conv2d: 2-3                       [1, 192, 56, 56]          [1, 128, 56, 56]          [3, 3]                    221,184
│    └─ReLU: 2-4                         [1, 128, 56, 56]          [1, 128, 56, 56]          --                        --
├─Sequentia

##### Case 2: 64 5x5 convolution filters

In [3]:
summary(InceptionNaiveModule(in_channels=192,
                             out_channels_1x1=32,
                             out_channels_3x3=128,
                             out_channels_5x5=64), 
                             input_size=(1, 192, 56, 56), 
                             col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
InceptionNaiveModule                     [1, 192, 56, 56]          [1, 416, 56, 56]          --                        --
├─Sequential: 1-1                        [1, 192, 56, 56]          [1, 32, 56, 56]           --                        --
│    └─Conv2d: 2-1                       [1, 192, 56, 56]          [1, 32, 56, 56]           [1, 1]                    6,144
│    └─ReLU: 2-2                         [1, 32, 56, 56]           [1, 32, 56, 56]           --                        --
├─Sequential: 1-2                        [1, 192, 56, 56]          [1, 128, 56, 56]          --                        --
│    └─Conv2d: 2-3                       [1, 192, 56, 56]          [1, 128, 56, 56]          [3, 3]                    221,184
│    └─ReLU: 2-4                         [1, 128, 56, 56]          [1, 128, 56, 56]          --                        --
├─Sequentia

We see that increasing the number of filters for 5x5 from 32 to 64 we see an increase of trainable parameters by ~$150,000$ parameters. To improve this module the authors of the paper took the concepts explained and experimented in the paper **Network in Network** paper and added it to the inception module.

The convolution filters with large receptive fields such as 3x3 and 5x5 do not get all the feature maps from the input through them. We now use **1x1** convolution filters called **reduce** filters which reduce the number of feature maps the module receives as input to a smaller number before passing to the number **3x3** and **5x5** filters.

The number of feature maps present in the output of max pooling layer is reduced using **1x1** filter before getting concatenated to the final output of the module. This module is what we call the true **Inception Module**.

This is what the Inception module will look like
```text
                           [Filter Concatenation]
                                     ▲
           ┌─────────────────┬───────┴───────┬─────────────────┐
           │                 │               │                 │
           │                 │               │                 │
[1x1 convolutions] [3x3 convolutions] [5x5 convolutions] [Pool Projection]
           ▲                 ▲               ▲                 ▲
           │                 │               │                 │
           │         [3x3 Reduction]   [5x5 Reduction]   [3x3 max pooling]
           ▲                 ▲               ▲                 ▲
           │                 │               │                 │
           └─────────────────┴───────┬───────┴─────────────────┘
                                     │
                             [Previous layer]
```

Below we will see an example of inception module where and see how many parameters are used as compared to the first version shown in the previous example.

Here **out_channels_3x3_reduce**, **out_channels_5x5_reduce** and **out_channels_pool_projection** represent the number of channels used by **1x1** filters to reduce the input feature maps for 3x3 and 5x5 convolution filters and reducing the number of channels as part of the output of the pooling layer.

In [4]:
summary(InceptionModule(in_channels=192,
                        out_channels_1x1=32,
                        out_channels_3x3_reduce=96,
                        out_channels_3x3=128,
                        out_channels_5x5_reduce=16,
                        out_channels_5x5=32,
                        out_channels_pool_projection=32), 
                        input_size=(1, 192, 56, 56), 
                        col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
InceptionModule                          [1, 192, 56, 56]          [1, 224, 56, 56]          --                        --
├─Sequential: 1-1                        [1, 192, 56, 56]          [1, 32, 56, 56]           --                        --
│    └─Conv2d: 2-1                       [1, 192, 56, 56]          [1, 32, 56, 56]           [1, 1]                    6,144
│    └─ReLU: 2-2                         [1, 32, 56, 56]           [1, 32, 56, 56]           --                        --
├─Sequential: 1-2                        [1, 192, 56, 56]          [1, 128, 56, 56]          --                        --
│    └─Conv2d: 2-3                       [1, 192, 56, 56]          [1, 96, 56, 56]           [1, 1]                    18,432
│    └─ReLU: 2-4                         [1, 96, 56, 56]           [1, 96, 56, 56]           --                        --
│    └─Conv2

We can see above that in the improved version of Inception module we are using ~$157,000$ trainable parameters when compared to ~$380,000$ parameters. This will look like why is a 157k parameter module better than 380k parameters. More parameters should mean possibility of extracting more information. It was experimented in the paper **Network in Network** and also implemented in **VGG** models that using 1x1 convolution layers helps compensate the reduction of feature maps. The advantage these reduction **1x1** convolution filters bring allow to build deeper models with less number of trainable parameters.

The next concept that GoogleNet introduces is **auxilary Classifiers**.

### auxilary Classifiers
We have seen till now that models being built going from AlexNet to VGG have been becoming deeper and larger models. As the depth grows we know that it becomes difficult to pass relevant information as part of gradients to the upstream initial layers. 

To handle this what the authors of the paper have done is they have introduced classifiers which take input from different layers spread across the entire feature extraction block and use them to help perform the classification task with the final classification block. Be it training or inference time the input gets propagated through the auxilary and the actual classifier block. At training time the output of the auxilary classifer and the actual classifier are weighted and the final loss is calculated upon which gradients are calculated and backpropagation takes place.

During inference or test time we propagate the input through all the classifiers but only use the information obtained in the main classifier

In Inception V1 the authors have used **2** auxliary classifiers and **1** final classifier block. For weights **0.3** is assigned to the output of auxilary classifier and **0.4** to the actual classifier.

## Model Architecture
Below table has information about how the Inception V1 model is structured. This table represents what the model looks like during the inference phase. During the training phase the 2 auxilary classifers are attached that take in output of inception 4a and inception 4d respectively. Below are few details you need to know about the model that is not provided in the table

1. The inception module cannot be used from the start of the model. The authors experimented with where the module can be placed and how does it impact the performance of the model. They found that placing the inception module at the very start of the model does not help and only degrades the models performance.
2. Inception module is not responsible for down sampling it happens outside of the module. To accomodate this 3x3 convolution filters have a padding of 1 and 5x5 convolution filters have a padding of 2 on all sides.
3. ReLU is the non linear activation function that is present after every convolution layer.
4. The Auxliary classifiers follow the format:
   1. Average Pooling Layer of kernel size 5 and stride 3
   2. Convolution Layer of kernel size 1 and stride 1
   3. Followed by ReLU
   4. The output of ReLU function gets flattened
   5. The output of flattening operation is 2048 features which pass through a linear layer to get projected to 1024 features
   6. These 1024 features get projected to the number of output classes

Below the table representing the model architecture you will see the summary of the model implemented. For 1000 classes you will see the model incorporates ~$13.37$ million parameters. The auxilary classifiers are responsible for half of it

### Model Architecture Table

| Layer Type          | Filter Attributes                                             | Input Shape      | 1×1 Conv Filter                | 3×3 Reduction Conv Filter | 3×3 Conv Filter | 5x5 Reduction Conv Filter | 5×5 Convolution Filter | 1x1 Max Pool Projections | Output Shape     |
| ------------------- | ------------------------------------------------------------- | ---------------- | ------------------------------ | ------------------------- | --------------- | ------------------------- | ---------------------- | ------------------------ | ---------------- |
|                     | (No of Filters, Kernel Size, Stride, Padding, Dilation, Bias) | (N,C,H,W)        | \-                        | \-                        | \-                        | \-                        | \-                        | \-                        | (N,C,H,W)                 |
| Conv 2D             | (64, 7, 2, 3, 1, True)                                        | (1 ,3, 224, 224)    | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 64,112,112)  |
| Max Pool            | (64, 3, 2, 1, 1, -)                                           | (1, 64, 112, 112)   | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 64, 56, 56)  |
| Local Response Norm | \-                                                            | (1, 64, 56, 56)  | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 64, 56, 56)  |
| Conv 2D             | (64, 1, 1, 0, 1, False)                                       | (1, 64, 56, 56)  | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 64, 56, 56)  |
| Conv 2D             | (64, 3, 1, 1, 1, False)                                       | (1, 64, 56, 56)  | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 192, 56, 56) |
| Local Response Norm | \-                                                            | (1, 192, 56, 56) | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 192, 56, 56) |
| Max Pool            | (192, 3, 2, 1, 1, -)                                          | (1, 192, 56, 56) | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 192, 28, 28) |
| Inception (3a)      | \-                                                            | (1, 192, 28, 28) | 64                             | 96                        | 128             | 16                        | 32                     | 32                       | (1, 256, 28, 28) |
| Inception (3b)      | \-                                                            | (1, 256, 28, 28) | 128                            | 128                       | 192             | 32                        | 96                     | 64                       | (1, 480, 28, 28) |
| Max Pool            | (480, 3, 2, 1, 1, -)                                          | (1, 480, 28, 28) | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 480, 14, 14) |
| Inception (4a)      | \-                                                            | (1, 480, 14, 14) | 192                            | 96                        | 208             | 16                        | 48                     | 64                       | (1, 512, 14, 14) |
| Inception (4b)      | \-                                                            | (1, 512, 14, 14) | 160                            | 112                       | 224             | 24                        | 64                     | 64                       | (1, 512, 14, 14) |
| Inception (4c)      | \-                                                            | (1, 512, 14, 14) | 128                            | 128                       | 256             | 24                        | 64                     | 64                       | (1, 512, 14, 14) |
| Inception (4d)      | \-                                                            | (1, 512, 14, 14) | 112                            | 144                       | 288             | 32                        | 64                     | 64                       | (1, 528, 14, 14) |
| Inception (4e)      | \-                                                            | (1, 528, 14, 14) | 256                            | 160                       | 320             | 32                        | 128                    | 128                      | (1, 832, 14, 14) |
| Max Pool            | (832, 3, 2, 1, 1, -)                                          | (1, 832, 14, 14) | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 832, 7, 7)   |
| Inception (5a)      | \-                                                            | (1, 832, 7, 7)   | 256                            | 160                       | 320             | 32                        | 128                    | 128                      | (1, 832, 7, 7)   |
| Inception (5b)      | \-                                                            | (1, 832, 7, 7)   | 384                            | 192                       | 384             | 48                        | 128                    | 128                      | (1, 1024, 7, 7)  |
| Avg Pool            | (1024, 7, 1, 0, 1, -)                                         | (1, 1024, 7, 7)  | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 1024, 1, 1)  |
| Flatten             | \-                                                            | (1, 1024, 1, 1)  |                                |                           |                 |                           |                        |                          | (1, 1024)        |
| Dropout (40%)       | \-                                                            | (1, 1024)        | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 1024)        |
| Linear              |                                                               | (1, 1024)        | \-                             | \-                        | \-              | \-                        | \-                     | \-                       | (1, 1000)        |

### Model Summary

In [5]:
summary(InceptionNetV1(1000), input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
InceptionNetV1                           [1, 3, 224, 224]          [1, 1000]                 --                        --
├─Sequential: 1-1                        [1, 3, 224, 224]          [1, 64, 112, 112]         --                        --
│    └─Conv2d: 2-1                       [1, 3, 224, 224]          [1, 64, 112, 112]         [7, 7]                    9,472
│    └─ReLU: 2-2                         [1, 64, 112, 112]         [1, 64, 112, 112]         --                        --
├─MaxPool2d: 1-2                         [1, 64, 112, 112]         [1, 64, 56, 56]           3                         --
├─LocalResponseNorm: 1-3                 [1, 64, 56, 56]           [1, 64, 56, 56]           --                        --
├─Sequential: 1-4                        [1, 64, 56, 56]           [1, 192, 56, 56]          --                        --
│    └─Conv2d: 2